# 02. 밸런스 + 뉴비 군집 분석 (팀 공식 통합 · 정리본)

`individual/01_balance_isolation_forest_shap.ipynb`와 `individual/02_planner_newbie_kmeans_clustering.ipynb`를 **하나로 합치고 실행 순서 문제를 고친** 정리본입니다. 원본 개별 노트북과 달리:

1. 한글 폰트 설정과 패키지 설치를 맨 위 한 번으로 통합 (트러블슈팅 #11, #12)
2. 정의되지 않은 변수를 참조하는 깨진 셀(`profile_long`, `death_stats` 관련)은 **제외** — 원본은 `individual/`에서 확인
3. 셀 순서를 '집계 → 이상치 탐지 → 시각화 → SHAP → 군집화' 논리 흐름대로 재배치

실행 전제: `01_character_mapping_and_dashboard_prep.ipynb`의 산출물 `EternalReturn_kakaogames_2024_character_added.csv`가 같은 작업 폴더에 있어야 합니다.


## 0. 공통 설정 (원본에 흩어져 있던 폰트·패키지 설정을 통합)

In [ ]:
# 원본 cell 10 + cell 14 주석을 통합 — 트러블슈팅 #11(한글 폰트), #12(shap 미설치) 해결
%pip install shap xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import shap
from xgboost import XGBClassifier

# 한글 폰트 (Colab). 로컬 환경이면 이미 설치된 한글 폰트로 교체하세요.
import subprocess, sys
try:
    subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum', '-q'], check=False)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 자동 설치 실패 — 로컬 환경이면 시스템 한글 폰트를 수동 지정하세요:', e)
plt.rcParams['axes.unicode_minus'] = False

CSV_PATH = "EternalReturn_kakaogames_2024_character_added.csv"
VERSION = 23
CONTAMINATION = 0.10
EXCLUDE = ["알론소", "일레븐"]


## 1. 밸런스팀 — 캐릭터별 집계 (패치 23.0 기준)

In [ ]:
df = pd.read_csv(CSV_PATH)
v = df[df["versionMajor"] == VERSION].copy()
total = len(v)

g = v.groupby("character_name_kr")
agg = pd.DataFrame({
    "표본수":       g.size(),
    "승률":         g["victory"].mean() * 100,
    "평균등수":     g["gameRank"].mean(),
    "평균생존시간": g["playTime"].mean(),
    "평균딜":       g["damageToPlayer"].mean(),
    "평균킬":       g["playerKill"].mean(),
})
agg["픽률"] = agg["표본수"] / total * 100
agg = agg.reset_index().rename(columns={"character_name_kr": "캐릭터"})
agg = agg[~agg["캐릭터"].isin(EXCLUDE)].reset_index(drop=True)
print(f"분석 캐릭터 {len(agg)}종 (v{VERSION})")
agg.head()

## 2. 밸런스팀 — Isolation Forest 다변량 이상치 탐지

In [ ]:
METRICS = ["승률", "픽률", "평균등수", "평균생존시간", "평균딜", "평균킬"]
Xs = StandardScaler().fit_transform(agg[METRICS])

iso = IsolationForest(n_estimators=400, contamination=CONTAMINATION, random_state=42)
iso.fit(Xs)
agg["비정상도"] = -iso.score_samples(Xs)
agg["이상치"] = (iso.predict(Xs) == -1)

wmean = agg["승률"].mean()
def direction(r):
    if not r["이상치"]:
        return "정상"
    return "너프 검토" if r["승률"] > wmean else "성능 개선 검토"
agg["점검방향"] = agg.apply(direction, axis=1)

후보 = agg[agg["이상치"]].sort_values("비정상도", ascending=False)
후보[["캐릭터", "승률", "픽률", "평균등수", "평균생존시간", "평균딜", "평균킬", "비정상도", "점검방향"]].round(2)

## 3. 밸런스팀 — 시각화 (픽률×승률 산점도)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
wm, pm = agg["승률"].mean(), agg["픽률"].mean()
ax.axhline(wm, ls="--", c="#999", lw=0.9)
ax.axvline(pm, ls="--", c="#999", lw=0.9)

col = {"정상": "#C7CFDB", "너프 검토": "#F97362", "성능 개선 검토": "#5AA9E6"}
for cls, c in col.items():
    sub = agg[agg["점검방향"] == cls]
    ax.scatter(sub["픽률"], sub["승률"], s=(90 if cls != "정상" else 45),
               c=c, edgecolor="white", linewidth=0.8, label=cls, alpha=0.9)
for _, r in agg[agg["이상치"]].iterrows():
    ax.annotate(r["캐릭터"], (r["픽률"], r["승률"]), fontsize=10, fontweight="bold",
                xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("픽률 (%)"); ax.set_ylabel("승률 (%)")
ax.set_title(f"다변량 이상치 탐지 — v{VERSION} 점검 후보", fontweight="bold")
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("anomaly_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. 밸런스팀 — SHAP 캐릭터별 심층 분석

`TARGET`을 바꿔가며 위 3단계에서 나온 이상치 후보들을 하나씩 심층 분석합니다. (원본 cell 15 — 지원가 특성 지표 포함 버전을 채택, cell 14의 축약 버전은 개념 설명용으로 `individual/`에만 보존)

In [ ]:
# 원본 cell 15에서 이미 위에서 설정한 폰트를 재사용하도록 중복 설정 라인만 제거
TARGET = "다르코"   # 분석할 캐릭터 이름만 바꾸면 재사용 가능

FEATURES = {
    "damageToPlayer":  "딜",
    "playTime":        "생존시간",
    "playerKill":      "킬",
    "healAmount":      "힐량",
    "teamRecover":     "팀회복",
    "protectAbsorb":   "보호막흡수",
    "playerAssistant": "어시스트",
    "monsterKill":     "몬스터킬",
}

df = pd.read_csv(CSV_PATH)
v = df[df["versionMajor"] == VERSION]
ch = v[v["character_name_kr"] == TARGET].copy()

feat = list(FEATURES.keys())
s = ch.dropna(subset=feat + ["victory"])
X = s[feat]
y = s["victory"]
print(f"{TARGET}: 표본 {len(X)}건 · 승률 {y.mean()*100:.1f}%")

model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      subsample=0.8, eval_metric="logloss", random_state=42)
model.fit(X, y)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

mean_abs = np.abs(shap_values).mean(0)
print(f"\n{TARGET} 승률 기여 요인 (SHAP):")
for i in np.argsort(mean_abs)[::-1]:
    corr = np.corrcoef(X.iloc[:, i], shap_values[:, i])[0, 1]
    direction = "승률↑" if corr > 0 else "승률↓"
    print(f"  {FEATURES[feat[i]]:10s} {mean_abs[i]:.3f} ({direction})")

X_kr = X.rename(columns=FEATURES)
plt.figure()
shap.summary_plot(shap_values, X_kr, show=False)
plt.title(f"{TARGET} — 승률 기여 요인 (SHAP)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"shap_{TARGET}.png", dpi=140, bbox_inches="tight", facecolor="white")
plt.show()


## 5. 기획자 — 뉴비 행동 기반 K-means 군집

트러블슈팅 #1(코발트 제외), #7(k=4 의도적 채택), #14(1게임 유저 제외) 참고.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

df = pd.read_csv("/content/EternalReturn_kakaogames_2024_character_added.csv")

# 일반 스쿼드만 사용
squad = df[df["matchingTeamMode"] == 3].copy()

squad["earlyLeave"] = (squad["playTime"] < 600).astype(int)
squad["survive10"] = (squad["playTime"] >= 600).astype(int)
squad["routeNotSelected"] = (
    (squad["routeIdOfStart"] == 0) | (squad["routeIdOfStart"] == -1)
).astype(int)


user_df = squad.groupby("userNum").agg(
    nickname=("nickname", "first"),
    games=("gameId", "count"),
    avg_playTime=("playTime", "mean"),
    earlyLeaveRate=("earlyLeave", "mean"),
    survive10Rate=("survive10", "mean"),
    winRate=("victory", "mean"),
    avg_rank=("gameRank", "mean"),
    avg_kill=("playerKill", "mean"),
    avg_teamKill=("teamKill", "mean"),
    avg_monsterKill=("monsterKill", "mean"),
    avg_craftUncommon=("craftUncommon", "mean"),
    avg_craftRare=("craftRare", "mean"),
    avg_craftEpic=("craftEpic", "mean"),
    avg_hyperLoop=("useHyperLoop", "mean"),
    avg_securityConsole=("useSecurityConsole", "mean"),
    avg_reconDrone=("useReconDrone", "mean"),
    routeNotSelectedRate=("routeNotSelected", "mean"),
    giveUpRate=("giveUp", "mean"),
    accountLevel=("accountLevel", "max"),
    rankPoint=("rankPoint", "max")
).reset_index()

# 1판 유저는 패턴 안정성이 낮아서 우선 제외
cluster_base = user_df[user_df["games"] >= 2].copy()

features = [
    "games",
    "avg_playTime",
    "earlyLeaveRate",
    "survive10Rate",
    "winRate",
    "avg_rank",
    "avg_kill",
    "avg_teamKill",
    "avg_monsterKill",
    "avg_craftUncommon",
    "avg_craftRare",
    "avg_craftEpic",
    "avg_hyperLoop",
    "avg_securityConsole",
    "avg_reconDrone",
    "routeNotSelectedRate",
    "giveUpRate"
]

X = cluster_base[features].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("=== 군집 수별 실루엣 점수 ===")
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    print(k, round(silhouette_score(X_scaled, labels), 4))

# 4개 군집으로 실행
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
cluster_base["cluster"] = kmeans.fit_predict(X_scaled)

summary = cluster_base.groupby("cluster").agg(
    users=("userNum", "count"),
    avg_games=("games", "mean"),
    avg_accountLevel=("accountLevel", "mean"),
    avg_rankPoint=("rankPoint", "mean"),
    avg_playTime=("avg_playTime", "mean"),
    earlyLeaveRate=("earlyLeaveRate", "mean"),
    survive10Rate=("survive10Rate", "mean"),
    winRate=("winRate", "mean"),
    avg_rank=("avg_rank", "mean"),
    avg_kill=("avg_kill", "mean"),
    avg_monsterKill=("avg_monsterKill", "mean"),
    avg_craftRare=("avg_craftRare", "mean"),
    avg_hyperLoop=("avg_hyperLoop", "mean"),
    avg_securityConsole=("avg_securityConsole", "mean"),
    routeNotSelectedRate=("routeNotSelectedRate", "mean"),
    giveUpRate=("giveUpRate", "mean")
).reset_index()

print("\n=== 군집별 요약 ===")
display(summary.sort_values("earlyLeaveRate", ascending=False))

cluster_base.to_csv("/content/user_behavior_clusters.csv", index=False, encoding="utf-8-sig")
summary.to_csv("/content/user_cluster_summary.csv", index=False, encoding="utf-8-sig")

## 참고: 제외된 셀
- 원본 cell 14 (SHAP 1차 축약 버전) → 개념 설명용, `individual/01_balance_isolation_forest_shap.ipynb`
- 원본 cell 18 (`profile_long` 참조, 정의 코드 누락) → `individual/02_planner_newbie_kmeans_clustering.ipynb`
- 원본 cell 19-20 (`death_stats` 참조, 정의 코드 누락) → `individual/03_misc_death_cause_fragment.ipynb`
